In [1]:
!pip install langchain-text-splitters boto3 python-dotenv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.6/140.6 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.8/14.8 MB 80.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 5.7 MB/s eta 0:00:00


In [6]:
import os
import json
import time
import boto3
from pathlib import Path
from dotenv import load_dotenv
from langchain_text_splitters import MarkdownHeaderTextSplitter, RecursiveCharacterTextSplitter

In [7]:

# ==========================================
# 1. CONFIGURACIÓN Y RUTAS (CAPA GOLD)
# ==========================================

load_dotenv()
AWS_ACCESS_KEY_ID = os.getenv("AWS_ACCESS_KEY_ID")
AWS_SECRET_ACCESS_KEY = os.getenv("AWS_SECRET_ACCESS_KEY")
AWS_SESSION_TOKEN = os.getenv("AWS_SESSION_TOKEN")
AWS_REGION = os.getenv("AWS_REGION")

S3_BUCKET_NAME = os.getenv("S3_BUCKET_NAME")
S3_PREFIX_SILVER = os.getenv("S3_PREFIX_SILVER")
S3_PREFIX_QUARANTINE = os.getenv("S3_PREFIX_QUARANTINE")
S3_PREFIX_GOLD =  os.getenv("S3_PREFIX_GOLD")

# Directorios locales temporales
DIR_SILVER_LOCAL = Path("/content/silver_descargas")
DIR_GOLD_LOCAL = Path("/content/gold_chunks")
DIR_SILVER_LOCAL.mkdir(parents=True, exist_ok=True)
DIR_GOLD_LOCAL.mkdir(parents=True, exist_ok=True)

In [8]:
s3_client = boto3.client(
    's3',
    aws_access_key_id=AWS_ACCESS_KEY_ID,
    aws_secret_access_key=AWS_SECRET_ACCESS_KEY,
    aws_session_token=AWS_SESSION_TOKEN,
    region_name=AWS_REGION
)

In [9]:
# ==========================================
# 2. CONFIGURACIÓN DE LOS SPLITTERS (OPTIMIZADA)
# ==========================================
# 1. Splitter Semántico (El Jefe: Respeta la jerarquía del documento)
headers_to_split_on = [
    ("#", "Header_1"),
    ("##", "Header_2"),
    ("###", "Header_3"),
]
markdown_splitter = MarkdownHeaderTextSplitter(headers_to_split_on=headers_to_split_on)

# 2. Splitter de Seguridad (Optimizado para RAG Moderno)
# 3000 caracteres (~600-800 tokens) captura secciones completas sin romper tablas o listas largas.
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=3000,
    chunk_overlap=300, # Solapamiento generoso para no perder el hilo entre ideas
    separators=["\n\n", "\n", ". ", " ", ""] # Prioriza cortar siempre al final de un párrafo
)

def generar_chunks_con_metadatos(texto, doc_id, seccion_num):
    """Toma el texto Markdown, lo divide y le inyecta metadatos críticos para el RAG."""
    chunks_finales = []

    # Paso 1: Cortar por subtítulos Markdown.
    # Si una subsección tiene menos de 3000 caracteres, NO se cortará más. Queda intacta.
    md_splits = markdown_splitter.split_text(texto)

    # Paso 2: Aplicar el Splitter de Seguridad SOLO a las subsecciones que sean gigantescas
    splits_finales = text_splitter.split_documents(md_splits)

    for i, split in enumerate(splits_finales):
        # Enriquecemos los metadatos base con los del documento
        metadatos = split.metadata
        metadatos["doc_id"] = doc_id
        metadatos["seccion"] = seccion_num
        metadatos["chunk_id"] = f"{doc_id}_sec{seccion_num}_{i}"

        # Añadimos longitud para analítica posterior si la necesitas
        longitud_caracteres = len(split.page_content)

        chunks_finales.append({
            "chunk_id": metadatos["chunk_id"],
            "doc_id": doc_id,
            "seccion": seccion_num,
            "jerarquia": metadatos,
            "longitud_chars": longitud_caracteres,
            "contenido": split.page_content
        })

    return chunks_finales

In [10]:
def obtener_archivos_s3(prefix):
    paginator = s3_client.get_paginator('list_objects_v2')
    archivos = []
    for page in paginator.paginate(Bucket=S3_BUCKET_NAME, Prefix=prefix):
        if 'Contents' in page:
            for obj in page['Contents']:
                if obj['Key'].endswith('.jsonl'):
                    archivos.append(obj['Key'])
    return archivos

In [11]:
# ==========================================
# 3. ORQUESTADOR MAESTRO (CHUNKING)
# ==========================================


def procesar_pipeline_chunking():
    print(" Iniciando Pipeline: Silver -> Gold (Chunking Semántico)")

    # Obtenemos todos los archivos procesados en TODAS las secciones de Silver
    archivos_silver = obtener_archivos_s3(S3_PREFIX_SILVER)

    if not archivos_silver:
        print(" No se encontraron archivos en la Capa Silver.")
        return

    # Agrupamos los archivos de Silver por documento
    # Estructura: {"FDS_1": ["silver/seccion_1/FDS_1.jsonl", "silver/seccion_2/FDS_1.jsonl"...]}
    documentos_agrupados = {}
    for s3_key in archivos_silver:
        nombre_archivo = os.path.basename(s3_key)
        doc_id = nombre_archivo.replace('.jsonl', '')
        if doc_id not in documentos_agrupados:
            documentos_agrupados[doc_id] = []
        documentos_agrupados[doc_id].append(s3_key)

    print(f" Se encontraron {len(documentos_agrupados)} documentos listos para chunking.\n")

    for doc_id, rutas_secciones in documentos_agrupados.items():
        print(f" Procesando Chunks para: {doc_id}")

        ruta_s3_gold = f"{S3_PREFIX_GOLD}chunks/{doc_id}_chunks.jsonl"
        ruta_local_gold = DIR_GOLD_LOCAL / f"{doc_id}_chunks.jsonl"

        # Verificador de idempotencia (Saltar si ya existe en Gold)
        try:
            s3_client.head_object(Bucket=S3_BUCKET_NAME, Key=ruta_s3_gold)
            print(f" Saltando: Ya existe en Gold.")
            continue
        except:
            pass # No existe, procedemos

        todos_los_chunks = []

        # Descargamos y unimos todas las secciones de este documento
        for ruta_s3 in rutas_secciones:
            response = s3_client.get_object(Bucket=S3_BUCKET_NAME, Key=ruta_s3)
            lineas = response['Body'].read().decode('utf-8').strip().split('\n')

            for linea in lineas:
                if not linea: continue
                datos_seccion = json.loads(linea)
                texto_seccion = datos_seccion.get("contenido", "")
                seccion_num = datos_seccion.get("seccion")

                # ¡MAGIA! Generamos los chunks
                chunks_seccion = generar_chunks_con_metadatos(texto_seccion, doc_id, seccion_num)
                todos_los_chunks.extend(chunks_seccion)

        # Guardamos todos los chunks consolidados del documento
        with open(ruta_local_gold, 'w', encoding='utf-8') as f:
            for chunk in todos_los_chunks:
                f.write(json.dumps(chunk, ensure_ascii=False) + '\n')

        # Subimos a S3 (Capa Gold)
        s3_client.upload_file(str(ruta_local_gold), S3_BUCKET_NAME, ruta_s3_gold)
        os.remove(ruta_local_gold)

        print(f"  Éxito. Creados {len(todos_los_chunks)} chunks semánticos.")

    print("\n PIPELINE GOLD COMPLETADO. ¡Datos listos para Vectorización!")

In [12]:
procesar_pipeline_chunking()

 Iniciando Pipeline: Silver -> Gold (Chunking Semántico)
 Se encontraron 23 documentos listos para chunking.

 Procesando Chunks para: Esmalte Uretano AR Comp. B
  Éxito. Creados 43 chunks semánticos.
 Procesando Chunks para: FDS 1 -pintura-antideslizante-para-canchas
  Éxito. Creados 38 chunks semánticos.
 Procesando Chunks para: FDS 10 - pintura-epoxi-poliamida-blanco-13243-10012925-10344427-hoja-de-seguridad-pdf
  Éxito. Creados 139 chunks semánticos.
 Procesando Chunks para: FDS 11  - Ecosphere Premium
  Éxito. Creados 90 chunks semánticos.
 Procesando Chunks para: FDS 12 - PINTURA ANTICORROSIVA GRIS 507 _ 10014333-10012454-10171712 _COL (Versión 3) (1)
  Éxito. Creados 144 chunks semánticos.
 Procesando Chunks para: FDS 13 -AVANTEX-FACHADA
  Éxito. Creados 43 chunks semánticos.
 Procesando Chunks para: FDS 14 -PINTURA-ACRILICA-PARA-TRÁFICO
  Éxito. Creados 40 chunks semánticos.
 Procesando Chunks para: FDS 15 - impra
  Éxito. Creados 49 chunks semánticos.
 Procesando Chunks para: 